# Transcribe the corpus

Transcribes every video in `/corpus/raw` with Whisper on the GPU. Nothing leaves the server.

**Before you start:** use a GPU workspace and check nobody else needs the GPU. This notebook is normally run once for the whole seminar.

Safe to re-run: videos with an existing result are skipped, so you can stop and continue at any time.

In [ ]:
import csv, json, os, subprocess, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from faster_whisper import WhisperModel
from tqdm.auto import tqdm

RAW = Path(os.environ.get("SEMINAR_RAW", "/corpus/raw"))
OUTPUT = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived"))
SHARED_MODEL = Path(os.environ.get("SEMINAR_MODELS", "/models")) / "faster-whisper-large-v3"

# None means detect the language per video. Set to "de" if all content is German.
LANGUAGE = None
# Limit for a trial run, for example 3. None processes everything.
MAX_VIDEOS = int(os.environ["SEMINAR_MAX_VIDEOS"]) if os.environ.get("SEMINAR_MAX_VIDEOS") else None

TRANSCRIPTS = OUTPUT / "transcripts"
VIDEO_SUFFIXES = {".mp4", ".mov", ".webm", ".mkv", ".m4v"}

## Check the setup

In [ ]:
assert torch.cuda.is_available(), "No GPU found. Restart your workspace and choose GPU."
print(torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print(f"GPU memory free: {free / 1024**3:.1f} of {total / 1024**3:.1f} GB")

TRANSCRIPTS.mkdir(parents=True, exist_ok=True)
assert os.access(TRANSCRIPTS, os.W_OK), f"Cannot write to {TRANSCRIPTS}."

## Find the videos

macOS helper files (names starting with `._`) are ignored. Each video is paired with the JSON file of the same name.

In [ ]:
videos = sorted(
    path for path in RAW.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES and not path.name.startswith(".")
)
if MAX_VIDEOS:
    videos = videos[:MAX_VIDEOS]


def result_path(video):
    return (TRANSCRIPTS / video.relative_to(RAW)).with_suffix(".json")


todo = [video for video in videos if not result_path(video).exists()]
print(f"{len(videos)} videos, {len(videos) - len(todo)} already done, {len(todo)} to transcribe")

## Load the model

Uses the shared model copy if available, otherwise downloads it to your home folder (about 3 GB).

In [ ]:
source = str(SHARED_MODEL) if SHARED_MODEL.exists() else "large-v3"
model = WhisperModel(source, device="cuda", compute_type="float16")
print("Loaded", source)

## Transcribe

Each result is saved as soon as it is finished. Failures are collected and don't stop the run.

In [ ]:
def has_audio(video):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=index", "-of", "csv=p=0", str(video)],
        capture_output=True, text=True, timeout=120,
    )
    return result.returncode == 0 and bool(result.stdout.strip())


def load_audio(video):
    # ffmpeg decodes the audio because faster-whisper 1.2.1 fails with the installed PyAV.
    pcm = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", str(video), "-vn", "-ac", "1", "-ar", "16000", "-f", "s16le", "-"],
        capture_output=True, timeout=600, check=True,
    ).stdout
    return np.frombuffer(pcm, np.int16).astype(np.float32) / 32768.0


def transcribe(video):
    segments, info = model.transcribe(load_audio(video), language=LANGUAGE, beam_size=5, vad_filter=True)
    segments = [{"start": round(s.start, 2), "end": round(s.end, 2), "text": s.text.strip()} for s in segments]
    return {
        "id": video.stem,
        "source": str(video.relative_to(RAW)),
        "model": "large-v3",
        "language": info.language,
        "language_probability": round(info.language_probability, 3),
        "duration": round(info.duration, 2),
        "text": " ".join(s["text"] for s in segments),
        "segments": segments,
    }


def save(path, record):
    path.parent.mkdir(parents=True, exist_ok=True)
    partial = path.with_suffix(".json.partial")
    partial.write_text(json.dumps(record, ensure_ascii=False, indent=1), encoding="utf-8")
    partial.rename(path)


errors = []
started = time.time()
for video in tqdm(todo, desc="Transcribing"):
    try:
        if not has_audio(video):
            errors.append({"source": str(video.relative_to(RAW)), "error": "no audio stream"})
            continue
        save(result_path(video), transcribe(video))
    except Exception as error:
        errors.append({"source": str(video.relative_to(RAW)), "error": f"{type(error).__name__}: {error}"})

print(f"Finished {len(todo) - len(errors)} of {len(todo)} in {(time.time() - started) / 60:.1f} minutes")
pd.DataFrame(errors, columns=["source", "error"]).to_csv(OUTPUT / "transcripts_errors.csv", index=False)
if todo and len(errors) == len(todo):
    raise RuntimeError(f"Every video failed. First error: {errors[0]}")
pd.DataFrame(errors, columns=["source", "error"])

## Build the table

One row per video, combining the transcript with the TikTok caption and metadata from the JSON file next to the video.

In [ ]:
def metadata(video):
    path = video.with_suffix(".json")
    if not path.exists():
        return {}
    post = json.loads(path.read_text(encoding="utf-8"))
    stats = post.get("stats") or {}
    return {
        "author": (post.get("author") or {}).get("uniqueId"),
        "created": pd.to_datetime(int(post["createTime"]), unit="s", utc=True) if post.get("createTime") else None,
        "caption": post.get("desc"),
        "plays": stats.get("playCount"),
        "likes": stats.get("diggCount"),
        "comments": stats.get("commentCount"),
        "shares": stats.get("shareCount"),
    }


rows = []
for video in videos:
    path = result_path(video)
    if path.exists():
        record = json.loads(path.read_text(encoding="utf-8"))
        rows.append({
            "id": record["id"],
            "contributor": video.relative_to(RAW).parts[0],
            "source": record["source"],
            **metadata(video),
            "language": record["language"],
            "duration": record["duration"],
            "transcript": record["text"],
        })

table = pd.DataFrame(rows)
table.to_csv(OUTPUT / "transcripts.csv", index=False, quoting=csv.QUOTE_NONNUMERIC)
print(len(table), "rows written to", OUTPUT / "transcripts.csv")
table.drop(columns=["caption", "transcript"], errors="ignore").head()

## Free the GPU

Run this when you are done, so others can use the GPU.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()
print("GPU memory released")